### Step 4: Model Preprocessing

Preprocess the data for modeling. Split the data into train, val, test datasets

In [ ]:
import pandas as pd

from sklearn.preprocessing import StandardScaler, OrdinalEncoder

def feature_engineered_filepath(res: str) -> str:
    return f"../data/003_feature_engineered/{res}_power_load.parquet"

def preprocess_filepath(res: str, split: str) -> str:
    return f"../data/004_preprocessed/{res}/{split}.parquet"

RESAMPLE_RESOLUTIONS = ["1min", "5min", "10min"]
VAL_DAYS  = 3
TEST_DAYS = 3
CATEGORICAL_COLS = ["workday", "time_of_day"]
DROP_COLS        = ["timestamp", "load"]

In [20]:
def preprocess(df: pd.DataFrame, scale: bool = True):

    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # 1. SORT FIRST (important for time series!)
    df = df.sort_values("timestamp")

    # 2. SPLIT FIRST (CRITICAL FIX)
    max_date = df["timestamp"].max()
    test_start = max_date - pd.Timedelta(days=TEST_DAYS)
    val_start  = test_start - pd.Timedelta(days=VAL_DAYS)

    df_train = df[df["timestamp"] <= val_start].copy()
    df_val   = df[(df["timestamp"] > val_start) & (df["timestamp"] <= test_start)].copy()
    df_test  = df[df["timestamp"] > test_start].copy()

    # 3. DROP NaNs AFTER SPLIT
    df_train = df_train.dropna()
    df_val   = df_val.dropna()
    df_test  = df_test.dropna()

    # 4. ENCODE (fit ONLY on train)
    enc = OrdinalEncoder()
    df_train[CATEGORICAL_COLS] = enc.fit_transform(df_train[CATEGORICAL_COLS])
    df_val[CATEGORICAL_COLS]   = enc.transform(df_val[CATEGORICAL_COLS])
    df_test[CATEGORICAL_COLS]  = enc.transform(df_test[CATEGORICAL_COLS])

    # 5. SPLIT X / y
    X_train, y_train = df_train.drop(columns=DROP_COLS), df_train["load"]
    X_val, y_val     = df_val.drop(columns=DROP_COLS), df_val["load"]
    X_test, y_test   = df_test.drop(columns=DROP_COLS), df_test["load"]

    # 6. SCALE (fit only on train)
    if scale:
        numeric_cols = [c for c in X_train.columns if c not in CATEGORICAL_COLS]

        scaler = StandardScaler()
        X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_val[numeric_cols]   = scaler.transform(X_val[numeric_cols])
        X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

    print(f"Train shape: {X_train.shape}\nVal shape: {X_val.shape}\nTest shape: {X_test.shape}")

    return X_train, y_train, X_val, y_val, X_test, y_test

# ── RUN ───────────────────────────────────────────────────────────────────────
for res in RESAMPLE_RESOLUTIONS:
    print(f"\nPreprocessing {res} dataset:")
    df = pd.read_parquet(feature_engineered_filepath(res))
    X_train, y_train, X_val, y_val, X_test, y_test = preprocess(df, scale=True)

    X_train = X_train.reset_index(drop=True)
    X_val   = X_val.reset_index(drop=True)
    X_test  = X_test.reset_index(drop=True)

    y_train = y_train.reset_index(drop=True).to_frame()
    y_val   = y_val.reset_index(drop=True).to_frame()
    y_test  = y_test.reset_index(drop=True).to_frame()

    X_train.to_parquet(preprocess_filepath(res, "X_train"), index=False)
    y_train.to_parquet(preprocess_filepath(res, "y_train"), index=False)
    X_val.to_parquet(preprocess_filepath(res, "X_val"), index=False)
    y_val.to_parquet(preprocess_filepath(res, "y_val"), index=False)
    X_test.to_parquet(preprocess_filepath(res, "X_test"), index=False)
    y_test.to_parquet(preprocess_filepath(res, "y_test"), index=False)


Preprocessing 1min dataset:
Train shape: (25644, 42)
Val shape: (4103, 42)
Test shape: (4316, 42)

Preprocessing 5min dataset:
Train shape: (5184, 42)
Val shape: (796, 42)
Test shape: (864, 42)

Preprocessing 10min dataset:
Train shape: (2592, 42)
Val shape: (369, 42)
Test shape: (432, 42)


In [21]:
X_train

,workday,lag_1min,lag_5min,lag_15min,lag_30min,lag_1hour,lag_6hour,lag_12hour,lag_1day,lag_1week,...,slope_30,slope_60,second,minute,hour,time_of_day,day,weekday,is_weekend,month
0,0.0,-0.376013,-0.376013,-0.425036,-0.424155,-0.186388,-0.410829,-0.613358,-0.624181,-0.563932,...,0.048070,-0.046466,0.0,-1.46385,-1.661325,3.0,-1.638356,0.399043,-0.707107,0.0
1,0.0,-0.324944,-0.324944,-0.376050,-0.407472,-0.437408,-0.385749,-0.623116,-0.591182,-0.659267,...,0.005872,-0.002103,0.0,-0.87831,-1.661325,3.0,-1.638356,0.399043,-0.707107,0.0
2,0.0,-0.310524,-0.310524,-0.324981,-0.425074,-0.232586,-0.410644,-0.354400,-0.571511,-0.538049,...,-0.008996,0.047517,0.0,-0.29277,-1.661325,3.0,-1.638356,0.399043,-0.707107,0.0
3,0.0,0.014431,0.014431,-0.310560,-0.376087,-0.424359,-0.232824,-0.652908,-0.569231,-0.393489,...,-0.014972,0.093955,0.0,0.29277,-1.661325,3.0,-1.638356,0.399043,-0.707107,0.0
4,0.0,-0.217011,-0.217011,0.014399,-0.325017,-0.407674,-0.430538,-0.719808,-0.586008,-0.503685,...,0.000384,0.145900,0.0,0.87831,-1.661325,3.0,-1.638356,0.399043,-0.707107,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2587,1.0,-0.503382,-0.503382,-0.353984,-0.427369,-0.053119,-0.613004,-0.408884,-0.448297,-0.537184,...,0.355208,-0.549219,0.0,-0.87831,1.661325,3.0,1.638356,-1.516365,-0.707107,0.0
2588,1.0,-0.458801,-0.458801,-0.503422,-0.497279,-0.458766,-0.611353,-0.327095,-0.496452,-0.777994,...,0.337357,-0.538439,0.0,-0.29277,1.661325,3.0,1.638356,-1.516365,-0.707107,0.0
2589,1.0,-0.400793,-0.400793,-0.458840,-0.354020,-0.329078,-0.616591,-0.336711,-0.473406,-0.628558,...,0.311394,-0.489617,0.0,0.29277,1.661325,3.0,1.638356,-1.516365,-0.707107,0.0
2590,1.0,-0.511380,-0.511380,-0.400830,-0.503460,-0.427572,-0.599149,-0.220276,-0.498992,-0.729730,...,0.306473,-0.409418,0.0,0.87831,1.661325,3.0,1.638356,-1.516365,-0.707107,0.0
